# 02 — EDA và pilot RQ-VAE

Notebook dùng embedding từ notebook 01 để chọn `hidden_dims` và `codebook_sizes` trước khi train Phase A đầy đủ. Mỗi cấu hình được train ngắn trên cùng một sample rồi so sánh reconstruction, codebook usage, perplexity, collision, số parameter và thời gian.

## 0. Cấu hình

In [ ]:
from pathlib import Path

PREPROCESSED_ROOT = None
OUTPUT_ROOT = None

GITHUB_REPOSITORY_URL = "https://github.com/nam-htran/VSF-MiniApp-Ecommerce.git"
GITHUB_BRANCH = "main"
REPOSITORY_ROOT = Path("/kaggle/working/vsf-miniapp-ecommerce-source")

SAMPLE_SIZE = 100_000
VALIDATION_PERCENT = 20
PCA_SAMPLE_SIZE = 50_000
PILOT_STEPS = 1_000
BATCH_SIZE = 1_024
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
EMBED_DIM = 32
SEED = 2026

CANDIDATES = [
    {"name": "baseline", "hidden_dims": [256, 128, 64], "codebook_sizes": [128, 64, 32]},
    {"name": "compact_codebook", "hidden_dims": [256, 128, 64], "codebook_sizes": [64, 32, 16]},
    {"name": "balanced_codebook", "hidden_dims": [256, 128, 64], "codebook_sizes": [64, 64, 64]},
    {"name": "larger_codebook", "hidden_dims": [256, 128, 64], "codebook_sizes": [256, 128, 64]},
    {"name": "shallow_mlp", "hidden_dims": [256, 128], "codebook_sizes": [128, 64, 32]},
    {"name": "narrow_mlp", "hidden_dims": [128, 64], "codebook_sizes": [128, 64, 32]},
    {"name": "deep_mlp", "hidden_dims": [256, 192, 128, 64], "codebook_sizes": [128, 64, 32]},
]

## 1. Chuẩn bị môi trường Kaggle

In [ ]:
import gc
import importlib.util
import json
import subprocess
import sys
import time

packages = []
for module, package in {
    "gin": "gin-config==0.5.0",
    "einops": "einops>=0.8.0",
    "huggingface_hub": "huggingface-hub>=0.25.0",
    "pyarrow": "pyarrow>=16.0.0",
    "sklearn": "scikit-learn>=1.4",
}.items():
    if importlib.util.find_spec(module) is None:
        packages.append(package)
if packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from tqdm.auto import trange

if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator before running the pilots.")

print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))

## 2. Tìm source và embedding notebook 01

In [ ]:
REPOSITORY_ROOT = Path(REPOSITORY_ROOT).expanduser().resolve()
if (REPOSITORY_ROOT / ".git").is_dir():
    subprocess.run(
        ["git", "-C", str(REPOSITORY_ROOT), "pull", "--ff-only", "origin", GITHUB_BRANCH],
        check=True,
    )
elif REPOSITORY_ROOT.exists():
    raise FileExistsError(f"Clone target is not a Git repository: {REPOSITORY_ROOT}")
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH, GITHUB_REPOSITORY_URL, str(REPOSITORY_ROOT)],
        check=True,
    )

SOURCE_ROOT = REPOSITORY_ROOT / "pi-recommendation/src"
if not (SOURCE_ROOT / "modules/rqvae.py").is_file():
    raise FileNotFoundError(f"RQ-VAE source not found: {SOURCE_ROOT}")
sys.path.insert(0, str(SOURCE_ROOT))

from modules.normalize import l2norm
from modules.quantize import QuantizeForwardMode
from modules.rqvae import RqVae


def has_embeddings(path):
    path = Path(path)
    return (path / "global_product_embeddings.f16.npy").is_file() and (
        path / "global_embedding_index.parquet"
    ).is_file()


if PREPROCESSED_ROOT is not None:
    PREPROCESSED_ROOT = Path(PREPROCESSED_ROOT).expanduser().resolve()
elif Path("/kaggle/working/preprocessed").exists():
    PREPROCESSED_ROOT = Path("/kaggle/working/preprocessed")
else:
    matches = list(Path("/kaggle/input").glob("**/global_product_embeddings.f16.npy"))
    PREPROCESSED_ROOT = matches[0].parent if matches else None

if PREPROCESSED_ROOT is None or not has_embeddings(PREPROCESSED_ROOT):
    raise FileNotFoundError("Notebook 01 embeddings were not found.")

if OUTPUT_ROOT is None:
    OUTPUT_ROOT = Path("/kaggle/working/rqvae-eda")
OUTPUT_ROOT = Path(OUTPUT_ROOT).expanduser().resolve()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("PREPROCESSED_ROOT:", PREPROCESSED_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)

## 3. EDA embedding

Kiểm tra shape, norm và intrinsic dimension gần đúng. PCA chỉ là tín hiệu tham khảo; lựa chọn cuối cùng dựa trên pilot RQ-VAE.

In [ ]:
embedding_path = PREPROCESSED_ROOT / "global_product_embeddings.f16.npy"
index_path = PREPROCESSED_ROOT / "global_embedding_index.parquet"
embeddings = np.load(embedding_path, mmap_mode="r")
product_index = pd.read_parquet(index_path, columns=["product_index", "product_id"])

if embeddings.ndim != 2 or len(embeddings) != len(product_index):
    raise ValueError("Embedding matrix and product index do not align.")
if not np.array_equal(product_index["product_index"].to_numpy(), np.arange(len(product_index))):
    raise ValueError("product_index must match embedding row order.")

rng = np.random.default_rng(SEED)
sample_size = min(SAMPLE_SIZE, len(embeddings))
sample_indices = rng.choice(len(embeddings), size=sample_size, replace=False)
sample = np.asarray(embeddings[sample_indices], dtype=np.float32)
sample_norms = np.linalg.norm(sample, axis=1)

print("Catalog products:", f"{len(embeddings):,}")
print("Embedding shape:", embeddings.shape)
print("Embedding dtype:", embeddings.dtype)
print("Sample norm min/mean/max:", sample_norms.min(), sample_norms.mean(), sample_norms.max())
print("Embedding file size GiB:", embedding_path.stat().st_size / 2**30)

In [ ]:
pca_rows = sample[rng.choice(len(sample), size=min(PCA_SAMPLE_SIZE, len(sample)), replace=False)]
pca_components = min(128, pca_rows.shape[1], len(pca_rows) - 1)
pca = PCA(n_components=pca_components, svd_solver="randomized", random_state=SEED)
pca.fit(pca_rows)
cumulative_variance = np.cumsum(pca.explained_variance_ratio_)

for threshold in [0.80, 0.90, 0.95]:
    reached = np.flatnonzero(cumulative_variance >= threshold)
    print(f"Components for {threshold:.0%} variance:", int(reached[0] + 1) if len(reached) else f"> {pca_components}")

plt.figure(figsize=(8, 4))
plt.plot(np.arange(1, len(cumulative_variance) + 1), cumulative_variance)
plt.axhline(0.90, color="tab:red", linestyle="--", linewidth=1)
plt.xlabel("PCA components")
plt.ylabel("Cumulative explained variance")
plt.title("Embedding intrinsic-dimension proxy")
plt.grid(alpha=0.2)
plt.show()

## 4. Pilot sweep

Mỗi candidate dùng cùng train/validation sample và seed. K-means initialization được tắt để pilot ngắn và so sánh công bằng; cấu hình train chính vẫn có thể bật lại.

In [ ]:
train_array, validation_array = train_test_split(
    sample,
    test_size=VALIDATION_PERCENT / 100,
    random_state=SEED,
)
train_tensor = torch.from_numpy(train_array)
validation_tensor = torch.from_numpy(validation_array)
device = torch.device("cuda")


def compute_loss(model, x, temperature=0.2):
    # Match the current RQ-VAE objective without invoking torch.compile.
    quantized = model.get_semantic_ids(x, gumbel_t=temperature)
    reconstructed = l2norm(model.decode(quantized.embeddings.sum(axis=-1)))
    reconstruction = ((reconstructed - x) ** 2).sum(axis=-1)
    return (reconstruction + quantized.quantize_loss).mean(), reconstruction.mean()


@torch.inference_mode()
def evaluate_model(model, data, codebook_sizes):
    model.eval()
    semantic_ids = []
    reconstruction_sum = 0.0
    for start in range(0, len(data), BATCH_SIZE):
        x = data[start:start + BATCH_SIZE].to(device)
        quantized = model.get_semantic_ids(x, gumbel_t=0.001)
        reconstructed = l2norm(model.decode(quantized.embeddings.sum(axis=-1)))
        reconstruction_sum += ((reconstructed - x) ** 2).sum().item()
        semantic_ids.append(quantized.sem_ids.cpu().numpy())

    semantic_ids = np.concatenate(semantic_ids)
    metrics = {"val_reconstruction": reconstruction_sum / len(data)}
    for layer, size in enumerate(codebook_sizes):
        _, counts = np.unique(semantic_ids[:, layer], return_counts=True)
        probabilities = counts / counts.sum()
        metrics[f"usage_{layer}"] = len(counts) / size
        metrics[f"perplexity_{layer}"] = float(np.exp(-(probabilities * np.log(probabilities)).sum()))

    _, cluster_sizes = np.unique(semantic_ids, axis=0, return_counts=True)
    metrics["unique_full_sid_rate"] = len(cluster_sizes) / len(semantic_ids)
    metrics["collision_rate"] = 1 - metrics["unique_full_sid_rate"]
    metrics["cluster_p50"] = float(np.quantile(cluster_sizes, 0.50))
    metrics["cluster_p90"] = float(np.quantile(cluster_sizes, 0.90))
    metrics["cluster_max"] = int(cluster_sizes.max())
    return metrics


def run_pilot(candidate):
    torch.manual_seed(SEED)
    model = RqVae(
        input_dim=train_tensor.shape[1],
        embed_dim=EMBED_DIM,
        hidden_dims=candidate["hidden_dims"],
        codebook_sizes=candidate["codebook_sizes"],
        codebook_kmeans_init=False,
        codebook_normalize=False,
        codebook_sim_vq=False,
        codebook_mode=QuantizeForwardMode.ROTATION_TRICK,
        commitment_weight=0.25,
        n_cat_features=0,
    ).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    generator = torch.Generator().manual_seed(SEED)
    started = time.time()
    model.train()

    progress = trange(PILOT_STEPS, desc=candidate["name"], leave=False)
    for step in progress:
        rows = torch.randint(len(train_tensor), (BATCH_SIZE,), generator=generator)
        x = train_tensor[rows].to(device)
        optimizer.zero_grad()
        loss, reconstruction = compute_loss(model, x)
        loss.backward()
        optimizer.step()
        if step % 100 == 0:
            progress.set_postfix(loss=f"{loss.item():.4f}")

    metrics = evaluate_model(model, validation_tensor, candidate["codebook_sizes"])
    metrics.update({
        "name": candidate["name"],
        "hidden_dims": candidate["hidden_dims"],
        "codebook_sizes": candidate["codebook_sizes"],
        "parameters": sum(parameter.numel() for parameter in model.parameters()),
        "seconds": time.time() - started,
    })
    del model, optimizer
    gc.collect()
    torch.cuda.empty_cache()
    return metrics

In [ ]:
results = pd.DataFrame([run_pilot(candidate) for candidate in CANDIDATES])
results["min_usage"] = results[["usage_0", "usage_1", "usage_2"]].min(axis=1)
results = results.sort_values("val_reconstruction").reset_index(drop=True)

display(results[[
    "name", "hidden_dims", "codebook_sizes", "parameters", "seconds",
    "val_reconstruction", "min_usage", "collision_rate",
    "cluster_p50", "cluster_p90", "cluster_max",
]])

## 5. So sánh và lưu kết quả

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].scatter(results["parameters"], results["val_reconstruction"])
axes[0].set_xlabel("Parameters")
axes[0].set_ylabel("Validation reconstruction")
axes[0].set_title("Model size vs reconstruction")

axes[1].scatter(results["collision_rate"], results["min_usage"])
axes[1].set_xlabel("Collision rate on validation sample")
axes[1].set_ylabel("Minimum codebook usage")
axes[1].set_title("Collision vs codebook health")

for _, row in results.iterrows():
    axes[0].annotate(row["name"], (row["parameters"], row["val_reconstruction"]), fontsize=8)
    axes[1].annotate(row["name"], (row["collision_rate"], row["min_usage"]), fontsize=8)
plt.tight_layout()
plt.show()

results_to_save = results.copy()
results_to_save["hidden_dims"] = results_to_save["hidden_dims"].map(json.dumps)
results_to_save["codebook_sizes"] = results_to_save["codebook_sizes"].map(json.dumps)
results_to_save.to_parquet(OUTPUT_ROOT / "rqvae_pilot_results.parquet", index=False)

report = {
    "sample_size": sample_size,
    "pilot_steps": PILOT_STEPS,
    "batch_size": BATCH_SIZE,
    "embed_dim": EMBED_DIM,
    "candidates": json.loads(results.to_json(orient="records")),
}
(OUTPUT_ROOT / "rqvae_pilot_report.json").write_text(
    json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("Saved pilot results to:", OUTPUT_ROOT)

## Cách chọn cấu hình

- Loại cấu hình có codebook usage thấp hoặc cluster cực lớn.
- Trong nhóm có reconstruction gần nhau, ưu tiên model ít parameter và chạy nhanh hơn.
- Collision không bắt buộc bằng 0 vì `sid_suffix` giải quyết exact identity; mục tiêu là prefix vẫn có cluster hữu ích.
- Chọn hai cấu hình tốt nhất để train đầy đủ với k-means initialization, sau đó mới chốt `hidden_dims` và `codebook_sizes` trong Gin.
- Pilot chỉ dùng item embedding; chất lượng cuối cùng vẫn phải được kiểm tra bằng ranking metrics ở Phase B.